## Stakaholder Analysis

This notebook is looking at the cleaned datasets to get an overview what would be interesting to further investigate.
The data is picked up after `data.py` stores it in `data/processed`.

Data is processed to answer the following questions with one chart each:<br>
    - How big is the customer base and how did it grow?<br>
    - Does a customer spend more over time or is growth only new customers?<br>
    - Where does the money go?<br>
    - How different are customers?<br>
    - How many customers are real users?<br>
    - Card in store or online?<br>
    - Where is the card used?<br>
    - Which currencies?<br>
    - Seasonality?<br>
    - Who are the top counterparts overall?<br>

 Further we want got get a first insight on:
    - Churn at a glance

The figures are exported to `fiures/eda`, and a short summary for the presentation to be found under each graph, with an overal insights summary at the very end.

In [ ]:
# load data, functions and libraries needed 
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# repo root, whether the notebook runs from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import plots, style
from src.data import load_customer_table, load_sow, load_sow_counterpart, monthly_active
from src.features import build_customer_features

style.apply()
pd.set_option("display.width", 200)

sow = load_sow()
cp = load_sow_counterpart()
feats = build_customer_features(load_customer_table())


## 1) How big is the How big is the customer base and how did it grow?
monthly_act = monthly_active(sow).to_frame()

ax = plots.active_customers(monthly_act)
plots.save_fig(ax.figure, "01_active_customers", "eda")
plt.show()

#### 1) monthly active customers interpretation
YPEAL was still growing between January 2021 and December 2023. Starting with around 690 active customers, over 3 years te number of active customers increased by factor 2.24.
After a very strong start beginning of 2021, many customers churned during summer - the reasons therefor may be many but only speculative at that stage.

In [ ]:
# 2) Does a customer spend more over time or is growth only new customers?

monthly_spend = sow.groupby("date").agg(spend=("total_amount", "sum")).join(monthly_active(sow))
monthly_spend["spend_per_active"] = monthly_spend["spend"] / monthly_spend["active_customers"]

# as trend is nearly linear after July 2021 until December 2023, a linear regression line will be added to the plot
trend_mask = (monthly_spend.index >= "2021-07-01") & (monthly_spend.index <= "2023-12-31")
df_trend = monthly_spend.loc[trend_mask]

#average spent per customer monthly
mean_trend = df_trend["spend_per_active"].mean()

x_nums = mdates.date2num(df_trend.index)
y_vals = df_trend["spend_per_active"].values

slope, intercept = np.polyfit(x_nums, y_vals, deg=1)
trend_line = slope * x_nums + intercept
monthly_slope = slope * 30.4375 # slope calculated over months, with a mean of 30.44 dd/mm

fig, ax = plt.subplots(figsize=(8, 3.4))

# real values over time
ax.plot(monthly_spend.index, monthly_spend["spend_per_active"], color=style.C[0], lw=2)


for x in (monthly_spend.index[0], monthly_spend.index[-1]):
    val = monthly_spend.loc[x, "spend_per_active"]
    ax.annotate(f"{val:.0f}", (x, val), xytext=(0, 6), textcoords="offset points", ha="center", color=style.INK2)


# dashed trend line
ax.plot(df_trend.index, trend_line, color=style.ACCENT, lw=1.8, ls="--", label="Trend (Jul 21 – Dec 23)")

points = [
    (df_trend.index[0], trend_line[0]),
    (df_trend.index[-1], trend_line[-1])
]
for x, val in points:
    ax.annotate(
        f"{val:.0f}",
        (x, val),
        xytext=(0, -20),
        textcoords="offset points",
        ha="center",
        color=style.ACCENT,
    )

# slope centered shown on trend line
mid_idx = len(df_trend) // 2
mid_date = df_trend.index[mid_idx]
mid_y = trend_line[mid_idx]

sign = "+" if monthly_slope >= 0 else ""
slope_label = f"Trend: {sign}{monthly_slope:.2f} CHF / mt."

ax.annotate(
    slope_label,
    (mid_date, mid_y),
    xytext=(0, 35),
    textcoords="offset points",
    ha="center",
    va="bottom",
    color=style.ACCENT,
    fontsize=9,
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.7)
)

plots.month_axis(ax)
style.clean(ax, "spend per active customer per month", "CHF / month")
plots.save_fig(fig, "02_spend_per_active", "eda")
plt.show()

print(f"average spent money / mt. per customer is : {mean_trend:,.2f} CHF")

#### 2) spend of all active customer interpretation

Customers spent approx. 1475 CHF / month per customer during the stable time of spending after July 2021.
The trend shows an increase of approx. +2.05 CHF / month per customer.
This graph let us interpret, that only small increase on the growth over this time is comming from each customer. The main point must be from new gained customers.

In [ ]:
# 3) Where does the money go?

cat_spend = sow.groupby("category")["total_amount"].sum()
cat_shares = (cat_spend / cat_spend.sum()) * 100

ax = plots.category_shares(cat_shares)
plots.save_fig(ax.figure, "03_where_the_money_goes", "eda")
plt.show()

#### 3) where the money goes
Almost half of the money is flowing into either shoppings and groceries, or in restaurant visits.

In [ ]:
# 4) How different are customers?

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

# Plot 1 - Log Spend
log_spend = np.log10(feats["total_amount"].clip(lower=1))
axes[0].hist(log_spend, bins=30, color=style.C[0], edgecolor="none")
style.clean(axes[0], "log10(spend)", "customers")

# Plot 2 - Log Transactions
log_tx = np.log10(feats["n_transactions"].clip(lower=1))
axes[1].hist(log_tx, bins=30, color=style.C[0], edgecolor="none")
style.clean(axes[1], "log10(transactions)", "customers")

# Plot 3 - Days Active
axes[2].hist(feats["days_active"], bins=30, color=style.C[0], edgecolor="none")
style.clean(axes[2], "days active", "customers")

fig.tight_layout()
plots.save_fig(fig, "04_customer_distributions", "eda")
plt.show()

#### 4) different customer groups?
Plot left shows that customers have spent between 0 and over 1'000'000 CHF in total. The share is nearly a gaussian bell curve, indicating that the spend is nearly normal distributed.
Middle plot shows most of the customers have used their card only once. After usage of approx. 3 and above, teh usage is regularly.
Plot on the right side, shows most of the customer are only few days aftive. After approx. 50 days of usage, the value of customers is slightly decreasing until at 900 days the number of customers still using it reaches 600 with a steep slope.

In [ ]:
# 5) How many customers are real users?

med_tx = feats["n_transactions"].median()
med_spend = feats["total_amount"].median()
pct_one_day = (feats["days_active"] <= 1).mean() * 100
pct_single_tx = (feats["n_transactions"] == 1).mean() * 100

metrics = [
    ("Median Transactions", f"{med_tx:.0f}"),
    ("Median Spend", f"{med_spend:.0f} CHF"),
    ("One-Day Customers", f"{pct_one_day:.1f}%"),
    ("Single-Tx Customers", f"{pct_single_tx:.1f}%")
]

fig, axes = plt.subplots(1, 4, figsize=(10, 2.2))
for ax, (title, val) in zip(axes, metrics):
    ax.axis("off")
    ax.text(0.5, 0.6, val, ha="center", va="center", fontsize=18, fontweight="bold", color=style.C[0])
    ax.text(0.5, 0.25, title, ha="center", va="center", fontsize=9, color=style.INK2)

fig.tight_layout()
plots.save_fig(fig, "05_real_user_stats", "eda")
plt.show()

#### 5) real users
The first two values show the how many transactions the overal customer does, and how much they spend.
Looking at the one-day customers, with approx. 9% the product seams not to convince on the first insights.
7.5% of all customers have only done one single transaction.

In [ ]:
# 6) Card in store or online?

df = sow.copy()
df["pos_amount"] = df["pos_perc"] * df["total_amount"]
df["ecom_amount"] = df["ecom_perc"] * df["total_amount"]

cat_summary = df.groupby("category")[["pos_amount", "ecom_amount", "total_amount"]].sum()

channel_shares = pd.DataFrame(index=cat_summary.index)
channel_shares["POS"] = (cat_summary["pos_amount"] / cat_summary["total_amount"])
channel_shares["E-Commerce"] = (cat_summary["ecom_amount"] / cat_summary["total_amount"])

# check if values of share are in percentage or 0-1, anyhow transformed to 0-100%
if channel_shares["POS"].mean() <= 1.0:
    channel_shares *= 100

channel_shares = channel_shares.sort_values("POS")

fig, ax = plt.subplots(figsize=(8, 4.5))
channel_shares[["POS", "E-Commerce"]].plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color=[style.C[0], style.C[2]],
    edgecolor="none",
)

style.clean(ax, "POS vs. e-commerce share per category", "% of spend")
ax.legend(["In-store (POS)", "E-commerce"], frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0)
ax.grid(axis="y", visible=False)
ax.set_xlim(0, 100)

plots.save_fig(fig, "06_pos_vs_ecom_share", "eda")
plt.show()

#### 6) shares of e-commers and in-store
The shares show that the card is used in-store where the customer needs to be on-site during the activity - e.g. paying a restaurant online makes less sense, as for the eating in a restaurant the customer needs to be there.

In [ ]:
# 7) Where is the card used?

cust_raw = load_customer_table()

country_cols = [c for c in cust_raw.columns if c.startswith("country_")]
country_totals = cust_raw[country_cols].sum()

# exclude "country_" for plots
country_totals.index = [c.replace("country_", "").upper() for c in country_totals.index]
country_totals = country_totals.sort_values(ascending=False)

# foreign use if more than one country
pct_foreign = (cust_raw["n_country"] > 1).mean() * 100

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(country_totals.index, country_totals.values, color=style.C[0])

style.clean(
    ax,
    f"top countries by transactions ({pct_foreign:.1f}% customers use card abroad)",
    "transactions",
)
plt.xticks(rotation=45, ha="right")

plots.save_fig(fig, "07_transactions_by_country", "eda")
plt.show()


#### 7) top countries where the card is used
Customers use their cards mostly in Switzerland, followed by many smaller values in different countries which are summarized as other. The rest is in Europe beside the United States.

In [ ]:
# 8) Which currencies?

cust_raw = load_customer_table()

currency_cols = [c for c in cust_raw.columns if c.startswith("cur_")]
currency_totals = cust_raw[currency_cols].sum()

# 2. Präfix 'cur_' entfernen, Währungscodes gross schreiben und absteigend sortieren
currency_totals.index = [c.replace("cur_", "").upper() for c in currency_totals.index]
currency_totals = currency_totals.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(currency_totals.index, currency_totals.values, color=style.C[0])

style.clean(ax, "card spend per currency", "total spend (CHF)")
plt.xticks(rotation=45, ha="right")

plots.save_fig(fig, "08_spend_by_currency", "eda")
plt.show()


#### 8) top currencies spend
The customer pay mainly CHF with their cards. The followers are also the currencies of the most visited countries.

In [ ]:
# 9) Seasonality?

df = sow.copy()
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

# amount per active customer
df["spend_per_cust"] = df["total_amount"] / df["n_customers"]

# Pivot-tabel: row = category, column = calender month (1-12)
pivot = df.pivot_table(
    index="category",
    columns="month",
    values="spend_per_cust",
    aggfunc="mean"
)

# indexing on category mean (1.0 = mena of the year per category)
cat_mean = pivot.mean(axis=1)
heatmap_data = pivot.divide(cat_mean, axis=0)

# column month labeling
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heatmap_data.columns = [month_names[m - 1] for m in heatmap_data.columns]

heatmap_data = heatmap_data.sort_index(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6.5))

sns.heatmap(
    heatmap_data,
    cmap="vlag",          # Diverging Colormap: below 1.0 cool, over 1.0 hot (centered at 1.0)
    center=1.0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "spend index (1.0 = category mean)"},
    ax=ax,
)

ax.set_title("seasonality: spend compared to mean of each category over all calender months", fontsize=11, fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plots.save_fig(fig, "09_seasonality_heatmap", "eda")
plt.show()


#### 9) seasonality
High-Volatility & Event-Driven Categories:
- `education`: Shows an extreme spike in May (2.50) and a secondary peak in October (1.47), reflecting enrollment and tuition payment deadlines.
- `tax`: Peaks in February (1.36), March (1.18), and January (1.09), driven by annual tax filing dates and prepayments in Q1.
- `salary`: Very irregular with highs in March (2.76), January (1.95), and December (1.77) (likely bonuses / 13th month salary), very low values in May/November (0.09), and a missing value in June.

Summer & Leisure Seasonality (Q3):
- `restaurants`: Reaches its annual high in July (1.33) and remains elevated from June through October (1.07–1.12), contrasting with a slow start to the year (January–March: 0.77–0.85).
- `holidays` & `transport`: Strongest in July (1.21 and 1.18 respectively), corresponding directly with the main summer vacation period.

Year-End & Holiday Spikes (Q4):
- `shopping`: Consistent around baseline for most of the year, then surges in November (1.12) (Black Friday) and December (1.18) (holiday shopping).
- `cash`: Climbs steadily toward the end of the year to reach peaks in November (1.11) and December (1.24).

Stable / Non-Seasonal Baselines:
- `groceries` (0.93–1.12), `health` (0.94–1.07), and `communication` (0.89–1.13) show minimal fluctuation around the 1.00 baseline throughout the year, confirming they represent inelastic daily essentials and fixed subscriptions.

In [ ]:
# 10) Who are the top counterparts overall?

# filter out the Other counterpart
cp_filtered = cp[~cp["top_counterpart"].astype(str).str.strip().str.lower().isin(["other", "others"])].copy()

top_cp = (
    cp_filtered.groupby("top_counterpart")["total_amount"]
    .sum()
    .sort_values(ascending=True)  # ascending=True für horizontalen Plot (Grösster oben)
    .tail(20)
)

scale = 1e6
top_cp_m = top_cp / scale

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(top_cp_m.index.astype(str), top_cp_m.values, color=style.C[0], edgecolor="none")

style.clean(ax, "top 20 counterparts by card spend", "total spend (M CHF)")
ax.grid(axis="y", visible=False)

for idx, (name, val) in enumerate(top_cp_m.items()):
    ax.annotate(
        f"{val:.1f}M",
        (val, idx),
        xytext=(4, 0),
        textcoords="offset points",
        va="center",
        color=style.INK2,
        fontsize=9,
    )

plots.save_fig(fig, "10_top_counterparts", "eda")
plt.show()


#### 10) top 20 countries spend
- Dominance of Swiss Supermarket Duopoly: Daily essentials lead the board, headed by `Coop` at 3.6M CHF and `Migros` at 2.9M CHF.
- Prominence of Fintechs & Payment Gateways: Digital financial services capture massive volume, with `Revolut` ranking third at 2.5M CHF, followed by `PayPal` at 1.1M CHF.
- Mid-Tier Tech, E-Commerce, & Mobility: Online retail, transit, hardware, and crypto platforms—specifically `Digitec Galaxus`, `SBB`, `Apple`, and `Binance`—tie closely around 0.7M CHF each.
- Discount Retail: Budget grocers (`Lidl`, `Denner`, `Aldi`) and `Amazon` form a steady cluster around 0.5M CHF each.
- Travel, Banking, & Leisure Long Tail: Traditional banks (`Kantonalbank`, `UBS`), travel providers (`Booking.com`, `Airbnb`, `Swiss`), and betting platforms (`Interwetten`) populate the lower end of the top 20 at roughly 0.4M CHF each.

Overall spend is heavily concentrated at the very top: the top 3 counterparts alone account for 9.0M CHF of total card volume.

In [ ]:
# 11) Churn at a glance



feats["churned_int"] = feats["churned"].map({True: 1, False: 0}).astype("Int64")

# calculate number and share inclusive <NA>
counts_series = feats["churned_int"].value_counts(dropna=False)
total_cust = len(feats)

n_active = int(counts_series.get(0, 0))
n_churn = int(counts_series.get(1, 0))
n_na = int(counts_series.get(pd.NA, 0))

pct_active = (n_active / total_cust) * 100
pct_churn = (n_churn / total_cust) * 100
pct_na = (n_na / total_cust) * 100

fig, ax = plt.subplots(figsize=(6.5, 4.0))

categories = ["Active (0)", "Churned (1)", "Unknown (<NA>)"]
rates = [pct_active, pct_churn, pct_na]
counts = [n_active, n_churn, n_na]
colors = [style.C[0], style.ACCENT, style.INK3 if hasattr(style, "INK3") else "grey"]

bars = ax.bar(categories, rates, color=colors, width=0.55)

for bar, pct, cnt in zip(bars, rates, counts):
    yval = bar.get_height()
    ax.annotate(
        f"{pct:.1f}%\n({cnt:,})",
        (bar.get_x() + bar.get_width() / 2, yval),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        color=style.INK2,
    )

ax.set_ylim(0, max(rates) * 1.25)
style.clean(ax, "customer churn & label availability", "% of total customers")
ax.grid(axis="x", visible=False)

plots.save_fig(fig, "11_customer_churn_rate", "eda")
plt.show()

feats["tenure_bucket"] = pd.qcut(feats["days_active"], q=4, labels=["Q1 (Low)", "Q2", "Q3", "Q4 (High)"])

bucket_churn = feats.groupby("tenure_bucket", observed=False)["churned_int"].mean() * 100

fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(bucket_churn.index.astype(str), bucket_churn.values, color=style.C[0], width=0.5)

for bar in bars:
    y = bar.get_height()
    ax.annotate(f"{y:.1f}%", (bar.get_x() + bar.get_width() / 2, y), xytext=(0, 4), textcoords="offset points", ha="center")

style.clean(ax, "churn rate by tenure bucket (days active)", "% churned")
plots.save_fig(fig, "11_churn_by_tenure", "eda")
plt.show()

#### 11) churn at a glance
30% of all customer's status is not known - as already introduced, it is the goal of this project, to predict their churn behaviour. The other 70% is split between both with 31.6 respectively 38.4%. 

## Summary of Stakeholder Analysis (EDA)

- Portfolio Growth Driven by Acquisition (Tasks 1–2):
    Total transaction volume and active customer counts expand consistently, with growth driven primarily by bringing in new users rather than large increases in spend per existing user.

- Spend Concentration by Category (Task 3):
    Turnover is concentrated in high-frequency everyday necessities, with groceries, general retail, and dining making up the dominant share of card spend.

- Bimodal Customer Engagement (Tasks 4–5):
    Customer behavior exhibits a distinct divide between a large group of trial/single-transaction accounts and an engaged core user base that drives the vast majority of cumulative spend.

- Channel Divergence (Task 6):
    Point-of-Sale (POS) transactions dominate daily physical routines (groceries, local transport), while e-commerce captures major shares in discretionary segments such as travel, entertainment, and digital services.

- Predominantly Domestic Usage (Tasks 7–8):
    While spend is heavily concentrated in the domestic home market, an identifiable active minority frequently uses foreign currencies (primarily EUR and USD).

- Clear Seasonal Cycles (Task 9):
    Spend per active customer shows predictable surges in summer for leisure (`restaurants` reaching 1.33 and `holidays` at 1.21 in July) and winter for retail (`shopping` at 1.18 and `cash` at 1.24 in December), contrasted with flat, inelastic baselines for groceries and health.

- Heavy Merchant Concentration (Task 10):
    Excluding generic placeholders (`"other"`), expenditure is strongly consolidated within the top 20 merchants, led by national supermarket chains and major mobility providers.

- Tenure-Driven Churn Dynamics (Task 11):
    Churn risk is heavily concentrated in the early lifecycle; users in the lowest `days_active` quartile drop off at high rates, while customers who cross the initial onboarding hurdle become sticky, long-term users.